## Understanding HCWF configurations and parameters

Each run is described by one YAML config file that lives with your project, outside the package. Start one from your own data with `hcwf-run init`, or copy the template `examples/config.yaml` from the HCWF repository, edit it, and give it to every run with `-p`:

```
hcwf-run init --obs /path/to/obs --model /path/to/model -o my_config.yaml   # detects the dataset details from your files
hcwf-run -p my_config.yaml --check                                          # validates and prints the plan, runs nothing
hcwf-run -p my_config.yaml                                                  # runs it
```

In a notebook, set the environment variable instead: `os.environ["HCWF_CONFIG"] = "/path/to/my_config.yaml"`. Without `-p` or `HCWF_CONFIG` HCWF stops and says so.

The file has sections: `observations`, `reference`, `models` (a list), `onset`, `units`, `blend`, `evaluation`, `metrics` and `output`, plus the top-level keys `project_name`, `region`, `mask` and `parallel`. Only what you must decide needs to be there; defaults are resolved at run time and the region supplies what it knows (onset definition, day boundary, box, climatology years). Every key, type, default and help line is in `hcwf/config/schema.py`, and unknown keys are errors with a "did you mean" suggestion. Each run writes its resolved config (every value and where it came from) next to its outputs.

This notebook explains the sections one by one.

---
### 1. Project, region and models

The top of the file names the project and the region, and lists the forecast models of the run.

```yaml
project_name: Demo run with the sample data
region: india          # a region or country of `hcwf-run onset regions`; null: no region, the whole domain of the files
models:
  - {name: IFS,  path: data/ifs,  variable: tp}
  - {name: NGCM, path: data/ngcm, variable: tp}
```

| Key | Type | Description |
| :--- | :--- | :--- |
| **project_name** | text | A descriptive title for your experiment, shown in the run banner. |
| **region** | text or `null` | Required. The target area. Known regions also supply the default onset definition, day boundary and climatology years (section 5). |
| **models** | list | The forecast models to evaluate, each with a `name` and the dataset keys of section 4. |

#### Key Concepts

##### 1. "Forecast categories"
All models of a run must belong to the same **forecast category**.
* **Example**: Do not mix deterministic models (single output) with probabilistic/ensemble models (multiple outputs) in the same run, as the verification metrics (like MAE vs. RPSS) differ significantly. Set `evaluation.probabilistic` to match.

##### 2. Choosing models on the command line
`hcwf-run -p my_config.yaml --model_list IFS` evaluates only the named models of the `models:` section.

#### Rules for YAML
* Lists and nested values use YAML: `[2015, 5, 1]` for a date, `[[1, 15], [16, 30]]` for windows. A single model is still a one-item list under `models:`.
* Quote text that YAML would read as something else, such as the file pattern `"{}.nc"`.
* `models:` is a list of mappings; each model carries its own `path`, `variable`, `units`, ... so there are no parallel lists to keep in step.

---
### 2. Observation data

This section defines how the system locates, reads and interprets the "ground truth" or observational data.

```yaml
observations:
  name: CHIRPS                 # optional label
  path: data/obs               # folder of the yearly files; ~ and ${ENV_VAR} work
  file_pattern: "{}.nc"        # {} is the year
  variable: RAINFALL
  units: mm                    # mm or m; read from the file's attribute if left out
  day_boundary: "00 UTC, labelled by start"   # the region's if left out
```

| Key | Type | Description |
| :--- | :--- | :--- |
| **name** | text | Identifier of the dataset (e.g., "IMD", "CHIRPS"). |
| **path** | path | Folder of the observation files. |
| **paths** | list | Instead of `path` and `file_pattern`: files, names or globs, any number per year (`years` keeps those of some years). |
| **file_pattern** | text | File naming pattern. `{}` is a placeholder for the year. Detected from the files if left out. |
| **variable** | text | The variable name for rainfall inside the file. Detected if it is the only gridded one. |
| **units** | `mm` or `m` | Units of the rainfall; `unit_cvt` is a multiplier to mm for files that declare none. |
| **dims** | mapping | Names of the file's dimensions if HCWF cannot tell them, e.g. `{time: TIME, lat: lat, lon: lon}`. |
| **day_boundary** | text | When the observed day starts and which date labels it. |

You can leave out what HCWF can detect: `hcwf-run datasets -p my_config.yaml` shows what each dataset resolved to, with the evidence for every detected value, and a value that cannot be detected reliably, or that the files contradict, is an error naming the key.

#### Implementation notes

* **Variable names**: `variable` must exactly match the variable name in the file.
* **Placeholders**: If your files are named by year (e.g., `1998.nc`), the pattern is `"{}.nc"`.
* **Units**: Use `units: m` if the source data is in meters; HCWF converts to mm.

#### Verification example:

In [ ]:
import os
from pathlib import Path

import yaml

import hcwf

config_path = Path("../examples/config.yaml").resolve()   # your own copy goes here
os.chdir(config_path.parents[1])                          # relative data paths in the example start at the repository root
config = yaml.safe_load(config_path.read_text())

# the file the config expects for one year
obs = config["observations"]
candidate = Path(obs["path"]) / obs["file_pattern"].format(2015)
print(f"{'found' if candidate.exists() else 'missing'}: {candidate}")

# the check HCWF does before a run, the same as `hcwf-run -p my_config.yaml --check`
plan = hcwf.check(str(config_path))
print("check passed:", plan.ok)

found: demo/data/obs/2015.nc
check passed: True


<br>

---

### 3. Reference model (benchmark)

This section sets the benchmark (a historical average or a simpler model) that your forecasts' skill is measured against. It is optional: climatology is the default.

```yaml
reference:
  model: climatology            # or conditional_climatology, or the name of a model
  climatology_cv: leave_one_year_out   # none | leave_one_year_out
```

| Key | Type | Description |
| :--- | :--- | :--- |
| **model** | text | `climatology` (default), `conditional_climatology` (the probability of onset on lead k given none by the issue date, from a KDE of past onsets) or a model name. |
| **climatology_cv** | text | `none` or `leave_one_year_out` (default): the year being scored is left out of its own climatology, for both climatology references (its onset is not one of the `climatology` members, the `conditional_climatology` KDE is refitted without it); a year outside `start_year_clim`..`end_year_clim` has nothing to leave out. The reference is always scored against the observed onsets the models are scored against, also for years outside the climatology years. |
| **path**, **file_pattern**, **variable**, **dims**, **accumulation**, **daily_boundary** | | Only for a reference that is a model: its data, read with its own settings, never a forecast model's. Defaults to the observations'. |

---

#### Why use a reference?

The benchmark allows you to calculate "Skill Scores." Instead of just knowing if your model is accurate, you learn if it is *better* than a simple baseline.

* **Climatology**: the long-term onset climatology of the observations of `evaluation.start_year_clim`..`end_year_clim`.
* **Another model**: for example last year's model, to see if a new one adds value.

#### Configuration tips

1.  **Shared directories**: A reference model's `path` defaults to the observations' folder.
2.  **Consistency**: Ensure the reference data end up in the same physical units as your observation and forecast data.
3.  **File pattern**: Like the observation section, the `{}` placeholder lets the script iterate through the years automatically.

<br>

---
### 4. Forecast models

Each entry of `models:` describes one forecast model. Because you may compare several models in one run, `models:` is a list.

```yaml
models:
  - name: IFS
    path: data/ifs
    file_pattern: "{}.nc"
    variable: tp
    units: m                  # mm or m; read from the file's attribute if left out
    lead_label: start         # start: lead k is dated init + k - 1; end: init + k
    accumulation: auto        # daily, since_init or auto (detected)
    daily_boundary: 03 UTC    # optional: cut days at a UTC time (sub-daily steps)
    members: All              # optional: ensemble members used
  - name: NGCM
    path: data/ngcm
    variable: tp
```

| Key | Type | Description |
| :--- | :--- | :--- |
| **name** | text | Model name used in output files (no spaces). |
| **path**, **file_pattern**, **paths**, **years** | | Where the files are, as for the observations. |
| **variable** | text | The rainfall variable (e.g., `tp` for total precipitation). |
| **units**, **unit_cvt** | | `mm` or `m`, or a multiplier to mm for files that declare no units. |
| **dims** | mapping | Names of the file's dimensions: `init_time`, `step`, `member`, `lat`, `lon`. |
| **lead_label** | text | Which date lead k gets. Detected against the observations if left out. |
| **accumulation**, **daily_boundary** | | Running totals (`since_init`) are differenced, sub-daily steps summed to days. |
| **dataset** | text | The id of an entry in a dataset file listed under `dataset_files:`; inline keys win. |

`hcwf-run init --dataset-template forecast` prints a blank entry.

#### Handling multiple models

Each model carries its own settings, so models with different variables, units or file names sit side by side:

* **Model 1 (IFS):** variable `tp` in meters, files `data/ifs/<year>.nc`.
* **Model 2 (NGCM):** variable `tp` in mm, files `data/ngcm/model_<year>.nc`.

```yaml
models:
  - {name: IFS,  path: data/ifs,  variable: tp, units: m}
  - {name: NGCM, path: data/ngcm, variable: tp, units: mm, file_pattern: "model_{}.nc"}
```

#### Limitations & potential errors

1.  **Units**: Different centers (ECMWF vs. NCEP) might use different default units. HCWF reads the file's `units` attribute and cross-checks it against the magnitudes; a contradiction is an error naming the key.
2.  **Whole-day leads**: HCWF reads lead steps as whole days. Aggregate sub-daily forecasts first, or set `daily_boundary`; `--check` hints at it.
3.  **A mix of one and many members**: keep deterministic and ensemble models in separate runs (section 1).

#### Unit consistency check
A common mistake in forecast evaluation is comparing "Precipitation Rate" (mm/day, kg/m2/s) with "Total Accumulation" (mm). Check your metadata so that all datasets reach a consistent daily-total basis in mm. `hcwf-run datasets -p my_config.yaml` shows the unit of every dataset and where it came from.

<br>

---
### 5. Region, masks and evaluation units

This section defines the spatial extent of your analysis. The tool provides independent ways to subset your data: the **region** (a box with defaults), a **mask** (which grid cells count) and **units** (evaluate on polygons instead of grid cells).

```yaml
region: ethiopia
mask:
  land: true
  country: Ethiopia
  cell_rule: centre            # or overlap_fraction (with overlap_fraction: 0.5)
units:
  shapefile: adm3.shp          # polygons: admin units, catchments, ...
  unit_id_col: ADM3_PCODE
```

| Key | Type | Description |
| :--- | :--- | :--- |
| **region** | text or `null` | Required. A target region or country such as "Ethiopia" (`hcwf-run onset regions` lists them; names are case-insensitive). It sets the box of cells read and, where known, the default onset definition, observation day boundary and climatology years (section 6). A region you add as a YAML file in `hcwf/onset/regions/` takes the defaults of the regions it is `part_of` (the example below does); with no `definitions` of its own or inherited it has none, so set `onset.definition` and `observations.day_boundary` yourself. `null` means no region: the whole domain of the files. |
| **mask** | mapping | Limits the evaluated cells: `land`, `country`, `file`, `condition`, `cell_rule`, `resolution`. The parts you give intersect. The mask is built on each dataset's own grid; rainfall is never regridded. |
| **units** | mapping | Evaluate on polygons or weighted cells instead of grid cells: `shapefile` (with `unit_id_col`, `unit_name_col`, `parent_col`), `weights`, `mode` (`cells`, `regrid`, `cell_transform`), `centroids`, `dissemination`, `box`, `outline`. Relative paths here start at the config file's folder. |
| **shpfile_dir**, **polygon** | path, bool | Region boundary drawn on the maps, and the region's polygon boundary as a mask (India has one). |

---

#### The ways to define space

1.  **Region**: Uses a named geographic area; its box says which cells are read.
2.  **Mask**: Keeps the cells that are land, inside a country, inside a mask file's shape, or above/below a field such as a DEM. Use it to drop ocean or other countries from a box.
3.  **Units**: Averages the rain of each dataset onto polygons (overlap-area weights, or your weights table), so that onsets are detected, scored and blended per polygon: admin units (ADM1-3), catchments. Benchmark outputs are written per unit as CSV, GeoJSON and NetCDF. `examples/units.yaml` runs on the sample data.

#### Configuration rules & limitations

* **Independence**: A region can be used with or without a mask, and a mask with or without units.
* **Data alignment**: Ensure your NetCDF grid resolution is sufficient for your chosen boundary. If your region is very small (e.g., a small district) and your model resolution is coarse (e.g., 1 degree), you may end up with very few data points for analysis.
* **Mask files**: A mask file must cover the data's extent; the run prints how it was aligned, and a mask that keeps no cell is an error.
* **Shapefiles**: A shapefile must have its `.shp`, `.shx` and `.dbf` (and `.prj`) components together.
* **Natural Earth data** for `land` and `country` is downloaded on first use; `scripts/prefetch_cartopy_data.py` fetches it for offline machines.

<br>

---
### 6. Rainy season/Monsoon Onset Criteria (Physics Logic)

This section says what counts as the onset of the rainy season or monsoon. The rule is a named **onset definition**: a short YAML file (for example ICPAC's), run identically on the observations and on every forecast. Every key below can be left out: it then takes the default of the `region` set in section 5, and HCWF stops with a message naming the key if the region has none.

```yaml
observations:
  day_boundary: "03 UTC, labelled by end"
onset:
  definition: moron_robertson
  threshold_years: [1998, 2024]
  forecast_history: none
```

| Key (in `onset:`) | Type | Description |
| :--- | :--- | :--- |
| **definition** | text | Registry id (e.g. `icpac`, `moron_robertson`) or path of a definition YAML file. Left out: the region's default. |
| **override**, **rule** | mapping | Change part of the definition (`override: {wet_day: at least 2 mm}`, merged like `extends`), or write a complete definition inline. |
| **season** | text | Season name, only for definitions with several seasons (`"1"` or `"2"` for derived seasons). |
| **threshold_years** | `[first, last]` | Years of observations that climatological thresholds are computed from. |
| **thresholds** | mapping | `{name: "path.nc:variable"}`: precomputed thresholds instead of computing them. |
| **threshold_source** | text | `observations` (default) or `model` (the model's own forecast climatology; mean-based thresholds only). |
| **lead_label** | text | `start`: lead k is dated init + k - 1. `end`: init + k. Left out: from `observations.day_boundary`. |
| **check_alignment** | bool | Off by default. When on, warns at the start of a run if observations and forecasts look shifted by a day (`hcwf-run -p my_config.yaml --check --alignment` runs it once). |
| **dry_spell_check** | bool | `false` drops the definition's dry-spell conditions (`reject if`). |
| **forecast_history** | text | `observed` (default): each forecast is searched together with the observed rain before its first lead. `none`: forecast leads only. |
| **backend** | text | `numba` (default), `reference` or `gpu`. All give identical results. |
| **percentage_threshold** | number | For ensembles: the fraction of members (0.0-1.0) that must have an onset for the ensemble to have one. |
| **explain_cells** | list | e.g. `[[9.0, 38.75]]`: for each cell and year, write an anatomy plot and a text log of the observed onset search to `output.figure_dir/explain/`. |

Further keys (`rain_resolution`, `missing`, `negatives`, `gaps`, `horizon`, `threshold_scale`, `threshold_table`, ...) make an onset search match another tool's exactly; they are written for you by `hcwf-run convert` (see `docs/migrating_from_onset_blending.md`).

---

#### Looking up definitions

```
hcwf-run onset list                 # definitions and their regions
hcwf-run onset describe icpac       # the rule in words
hcwf-run onset regions              # regions and their defaults
hcwf-run onset schematic icpac --png icpac.png
```

A rule that differs a little from a registry one goes in `onset.override`, or in your own YAML file with `extends: <id>` and only the changed keys; set `onset.definition` to its path. See `docs/onset_in_hcwf.md` for details.

#### What a definition contains

Every definition has the same pieces:
1. **Wet day**: the daily total that counts as wet (e.g. at least 1 mm).
2. **Season**: the dates the search runs over.
3. **Trigger**: a window of days (e.g. 5 days) with conditions: the first day is wet, the window total reaches an amount, and so on.
4. **Vetoes** (`reject if`): a dry run or dry window within some span after the candidate cancels it (the "false start" filter).
5. **Thresholds**: amounts that vary by place, such as the local mean 5-day total, computed from `threshold_years`.

#### Spatial thresholds

Rainfall varies strongly by geography (e.g. the Western Ghats vs. Rajasthan). Definitions that use a threshold such as the local mean 5-day total get one value per grid point (or unit), computed from the observations of `threshold_years` and cached (in `runs/.cache`, or `.onset_cache` inside an explicit `output.data_dir`; `hcwf-run cache info` lists the caches). Definitions with fixed amounts (e.g. 20 mm) use the same number everywhere.

#### Working with ensembles
If you are using probabilistic models (ensembles), `onset.percentage_threshold` is key.
* If set to `0.5`, the ensemble onset is only recorded for a grid point when **50% or more** of the members have an onset.

#### Old-style configs
The old flat onset keys (`wet_init`, `wet_threshold`, `wet_spell`, `dry_spell`, `dry_extent`, `thresh_file`, ...) are no longer read and a config that has one is refused. `hcwf-run convert old.yaml -o new.yaml` writes the equivalent `onset.rule` (see `docs/onset_in_hcwf.md`, "Converting the old flat keys").

<br>

#### Application Example: Translating Scientific Definitions to Config Settings

To see how a published definition becomes a config setting, take the monsoon onset of Moron and Robertson (2014).

##### The Objective
> "Identify the first wet day (**≥1 mm**) of the first **5-day** wet sequence from **April 1st** that receives at least the **climatological 5-day amount**, without being followed by a **10-day dry spell** (receiving **less than 5 mm**) in the following **30 days**."

##### The definition

This is the registry definition `moron_robertson` (`hcwf-run onset describe moron_robertson` prints it in words):

```yaml
wet_day: at least 1 mm
season:
  from: 04-01
  to: 10-31
thresholds:
  wet_spell_amount:
    mean total over: 5 days
    during: 05-01 to 09-30
onset:
  window: 5 days
  require:
    - first day: wet
    - total: at least wet_spell_amount
  reject if:
    - dry window: 10 days with less than 5 mm
      within: 30 days after window
```

| Scientific Requirement | Where it is set |
| :--- | :--- |
| First wet day (≥1 mm) | `wet_day: at least 1 mm` and `first day: wet` |
| 5-day wet sequence | `window: 5 days` |
| Climatological amount | threshold `wet_spell_amount`, computed from observations of `threshold_years` |
| 10-day dry spell, less than 5 mm | `dry window: 10 days with less than 5 mm` |
| Following 30 days | `within: 30 days after window` |
| April 1st start | `season: from: 04-01` |

##### The Configuration

In your run file you only name the definition (and, outside a known region, the other section 6 settings):

```yaml
observations:
  day_boundary: "03 UTC, labelled by end"
onset:
  definition: moron_robertson
  threshold_years: [1998, 2024]
```

`evaluation.start_date` / `end_date` (section 7) choose the years and the forecast init dates; the onset season comes from the definition.

##### Visualizing the Logic Flow

1.  **Search Start**: the search scans the season from 1 April.
2.  **The Trigger**: a candidate day must be wet (≥ 1 mm).
3.  **Sustainability**: the 5 days from the candidate must total at least the local `wet_spell_amount`.
4.  **The "False Start" Check**: a 10-day window with less than 5 mm in total, starting within the 30 days after the 5-day spell, cancels the candidate. The search continues with the next day.  


##### Pro-Tip: the dry-spell check and short forecasts
The 5-day trigger plus the 30-day veto means this rule reads 35 days from each candidate. HCWF shortens the forecast lead window so that a forecast never ends before the rule can be decided. For short forecasts, set `onset.dry_spell_check: false` to drop the veto.

##### Below is an example with dummy data and time series plot:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup synthetic data (60 days) - Adjusted for a smaller display
np.random.seed(42)
days = np.arange(1, 61)
rain = np.random.gamma(2, 2, 60) 

# Insert False Start (Day 10)
rain[10:15] = [5, 12, 8, 10, 5]   
rain[15:26] = 0.8                # Followed by a long dry spell

# Insert True Onset (Day 40)
rain[40:45] = [12, 18, 10, 22, 14] 
rain[45:60] = np.random.uniform(5, 12, 15)

# 2. Plotting with reduced figsize (approx half-size)
plt.figure(figsize=(6, 4))
plt.bar(days, rain, color='skyblue', width=0.8, label='Rainfall')

# Threshold lines
plt.axhline(y=1, color='red', linestyle='--', linewidth=1, label='wet day (1 mm)')
plt.axhline(y=5, color='orange', linestyle='--', linewidth=1, label='dry-window limit (5 mm)')

# Simplified Annotations
plt.annotate('False Start\n(Dry spell within 30d)', xy=(12, 12), xytext=(5, 20),
             arrowprops=dict(arrowstyle='->', color='black'))
plt.annotate('True Onset', xy=(40, 18), xytext=(42, 26),
             arrowprops=dict(arrowstyle='->', color='green'))

plt.title("Rainy season Onset Logic Simulation", fontsize=10)
plt.xlabel("Days since April 1st", fontsize=9)
plt.ylabel("Rainfall (mm)", fontsize=9)
plt.legend(fontsize=8, loc='best')
plt.grid(axis='y', alpha=0.2)
plt.tight_layout() # Ensures labels aren't cut off in small plots
plt.show()

#### Summary of the "Onset Loop"
For every grid point (and, for forecasts, every init and member), the search takes candidate days in order and stops at the first one that passes:

1.  **Wet day?** Is the candidate day wet (`first day: wet`)?
2.  **Trigger**: do the days of the window meet the requirements (total at least the threshold, number of wet days, ...)?
3.  **Veto check**: look at the span after the candidate. Is there a dry run or dry window that the definition rejects?
    * **If found**: the candidate was a false start. Move to the next day.
    * **If not found**: the candidate day is recorded as the **onset date**.

Every result has a status: `ONSET`, `NOT_FOUND`, `INSUFFICIENT_DATA` (missing data decided it), and for forecasts `PRIOR` (the observations had already confirmed the onset before the init).

> **Tutorial Note**: Comparing definitions, or switching the dry-spell check on and off, is the most effective way to see how sensitive the onset is for a region (e.g. Kerala vs. Northeast India).

<br>

---
### 7. Temporal settings (dates & years)

This section manages the time windows for both your forecast evaluation and your climatological baseline. It also handles the logic for "initialization days," which is crucial for sub-seasonal to seasonal (S2S) models that only run on specific days of the week.

```yaml
evaluation:
  start_date: [2013, 5, 1]
  end_date: [2015, 10, 31]
  start_year_clim: 1998
  end_year_clim: 2024
  # years: '2013:2015'              # optional: only these evaluated years
  # climatology_years: '2000:2020,2022'
  init_days: [0, 3]
  date_filter_year: 2024
```

| Key (in `evaluation:`) | Type | Description |
| :--- | :--- | :--- |
| **start_date / end_date** | `[year, month, day]` | The full range for evaluation; the month/day bound the forecast init dates. |
| **start_year_clim / end_year_clim** | integer | The year range used for the climatology. Left out: the region's. |
| **years** / **climatology_years** | list or text | Optional explicit year lists, such as `'2000:2020,2022'` or `[2000, 2002]`; they may skip years but must lie within the date ranges above. Left out, every year of those ranges is used. |
| **init_days** | list | Days of the week the model is initialized (0=Monday, 3=Thursday). |
| **date_filter_year** | integer | A reference year used to map calendar dates to specific weekdays. |

---

#### Climatology vs. evaluation period

It is vital to distinguish between these two timeframes:
1.  **Evaluation period**: The specific days/years you are benchmarking the model's accuracy (e.g., 2013-2015).
2.  **Climatological period**: The years used to define "normal" rainfall. Usually, this is a much longer period (e.g., 30 years). It comes from the region (India 1991-2020, Ethiopia 1998-2024) unless you set `start_year_clim` and `end_year_clim`; the observation files must cover it, and `--check` hints when it is short.
3.  The thresholds of an onset definition use `onset.threshold_years` (section 6), which defaults to the region's climatology years.

#### The initialization filter

Most S2S forecast models (like the IFS) do not run every day. They might only run twice a week.
* **`init_days: [0, 3]`**: This tells the script to only look for forecast data initialized on **Mondays** and **Thursdays**.
* **`date_filter_year`**: Because the date of a "Monday" changes every year, the system uses a reference year (e.g., 2024) to lock in the `MM/DD` combinations. For example, if May 6th was a Monday in 2024, the script will look for May 6th in 2013, 2014, and 2015 as an initialization candidate.

#### Limitations & tips

* **Leap years**: The system automatically handles February 29th, but ensure your `start_date` and `end_date` cover the full monsoon season (typically May through October for the Indian Monsoon).
* **One init day**: A list with one item is fine: `init_days: [0]`.

<br>

---
### 8. Evaluation and verification parameters

This section defines the "resolution" of your evaluation in terms of lead time. It determines how far into the future the model is tested and how much "wiggle room" (tolerance) is allowed for a forecast to be considered correct. Since weather forecasts naturally degrade over time, these parameters allow you to analyze model skill at different lead times (how far in advance the forecast was made).

```yaml
evaluation:
  verification_windows: [[1, 15], [16, 30]]
  tolerances: [3, 5]
  max_forecast_day: 30
  day_bins: [[1, 5], [6, 10], [11, 15], [16, 20], [21, 25], [26, 30]]
```

| Key (in `evaluation:`) | Type | Description |
| :--- | :--- | :--- |
| **verification_windows** | list of `[start, end]` | Specific ranges of days after initialization to evaluate (e.g., Week 1 vs. Week 2). |
| **tolerances** | list | The "margin of error" in days allowed in a verification window, one per window. If the model predicts onset 3 days late, it is a hit if the tolerance is at least 3. |
| **max_forecast_day** | integer | The last lead day used (e.g., 15 or 30 days). |
| **day_bins** | list of `[start, end]` | Smaller sub-windows used for grouping probabilistic statistics and plotting. |
| **probabilistic**, **members** | bool, list or text | Evaluate ensembles (section 9). |

On the command line the same keys are overrides: `--verification_window_list '1,15 16,30'`, `--tolerance_days_list 3 5`, `--max_forecast_day 15`.

#### Understanding verification windows vs. bins

While they look similar, they serve different purposes:

1.  **Verification windows**: These are the "macro" windows. You might want to see how the model performs in the first two weeks (`[1, 15]`) versus the second two weeks (`[16, 30]`). The model's skill is usually much higher in the first window.
2.  **Day bins**: These are the "micro" windows for grouping **probabilistic forecast** statistics. They are used to group daily data into chunks (like 5-day intervals) for metric statistics and plots.

#### The tolerance factor

`tolerances` is critical for calculating "Hit Rates."
* A tolerance of **0** means the forecast must match the observation exactly.
* A tolerance of **3** means if the observed onset is June 5th, any forecast between June 2nd and June 8th is considered a "Hit."

#### Constraints & limitations

* **Alignment**: The leads that can be searched depend on the length of your forecast files and on how many days the onset definition reads from a candidate day. HCWF prints a `Lead window` line at the start of a run and shortens `max_forecast_day`, the verification windows, tolerances and bins when needed. If no lead is usable it stops.
* **Overlap**: Ensure your `day_bins` fit within your `max_forecast_day`. For example, if `max_forecast_day` is 15, your bins shouldn't go up to 20.
* **Nesting**: Windows and bins are lists of `[start, end]` pairs: `[[1, 15], [16, 30]]`.

<br>

---
### 9. Metrics selection

This section defines the statistical "grading system" for your models. It allows you to toggle specific performance metrics on or off depending on whether you are evaluating a single forecast (deterministic) or a collection of forecasts (probabilistic/ensemble). The flags act as switches for the statistical analyses the pipeline will perform. There are two categories: **Deterministic** (focused on error and accuracy) and **Probabilistic** (focused on uncertainty and reliability).

```yaml
metrics:
  FAR: true
  MAE: true
  MR: true
  BS: true
  RPS: true
  AUC: true
  Reliability: true
  skill_score: true
evaluation:
  probabilistic: false
  members: All
```

| Key | Type | Description |
| :--- | :--- | :--- |
| **metrics.FAR** | bool | **False Alarm Ratio**: How often the model predicted an onset that didn't happen. |
| **metrics.MAE** | bool | **Mean Absolute Error**: The average number of days the forecast was off. |
| **metrics.MR** | bool | **Miss Rate**: How often the monsoon started but the model failed to predict it. |
| **metrics.BS, RPS, AUC, Reliability** | bool | Probabilistic scores: Brier, ranked probability, area under the ROC curve, reliability data. |
| **metrics.skill_score** | bool | Skill relative to the reference of section 3. |
| **evaluation.probabilistic** | bool | Set to `true` if you are evaluating an **ensemble** model. |
| **evaluation.members** | list or text | The ensemble members to include. |

#### Deterministic vs. probabilistic evaluation

The metrics you choose depend heavily on the nature of your forecast:

1.  **Deterministic metrics (`probabilistic: false`)**:
    Best for single-trajectory models (e.g., a single high-resolution run). These metrics measure the direct distance between the forecast date and the observed date.

2.  **Probabilistic metrics (`probabilistic: true`)**:
    Used when you have multiple versions of the same forecast (ensembles).
    * **Brier Score (BS)**: Measures the accuracy of probability forecasts.
    * **Ranked Probability Score (RPS)**: Evaluates the entire probability distribution.
    * **Reliability**: Generates data for reliability diagrams to see if the model is "overconfident" or "underconfident."

#### Configuration logic

* **The member list**: If `probabilistic: true`, `members` can be a list of integers such as `[1, 5, 7, 9]`, a text range such as `"1-5"`, or `All` (the default), which takes every member in the data.

* **Skill scores**: `skill_score: true` calculates how much better your forecast is compared to the `reference` of section 3.

* **Resource management**: Calculating probabilistic metrics like **AUC** and **Reliability** is computationally expensive. If you are processing a very large geographic region, turning these off can speed up the execution during the testing phase.

<br>

---
### 10. Output and graphics settings

This section of the configuration determines how your results are stored and visualized. It allows you to toggle between raw data exports (NetCDF/CSV) and various scientific visualization styles. You can choose to save raw processed data for further use in GIS software or generate publication-ready figures directly within the pipeline.

```yaml
output:
  data_dir: output           # leave out for a run folder: runs/<name>/ next to this file
  figure_dir: figures
  save_fig: true
  save_csv_score: true
  plot_heatmap_bss_auc: true
  plot_reliability: false
```

| Key (in `output:`) | Type | Description |
| :--- | :--- | :--- |
| **data_dir / figure_dir** | path | Folders where data files and images will be saved. With `data_dir` left out, a run writes to `runs/<region>_<definition>_<years>_<fingerprint>/` next to the config file; the same config is refused unless `--overwrite`. |
| **save_fig** | bool | Master switch to enable or disable all image generation. |
| **save_nc_spatial_far_mr_mae**, **save_nc_climatology** | bool | Exports processed spatial metrics (MAE, FAR, MR) and the climatological onset as NetCDF files. |
| **save_csv_score** | bool | Exports tabular skill scores, useful for external analysis in Excel or Pandas. |
| **save_onsets** | bool | Writes the gridded onset dates of every source as NetCDF (`hcwf-run export` does this without scoring). |
| **plot_...** | bool | The individual figures, below. |

---

#### Visualization options

The pipeline offers several specialized meteorological plots. Setting these to `true` generates them automatically:

* **Spatial maps** (`plot_spatial_far_mr_mae`, `plot_climatology_onset`): Visualizes errors (MAE, FAR, MR) or Climatology onset dates across the geographic grid.

* **Heatmaps** (`plot_heatmap_bss_auc`, `plot_panel_heatmap_error`, `plot_panel_heatmap_skill`): Excellent for comparing multiple models or different lead times at a glance. "Panel" heatmaps group several metrics into a single figure.

* **Reliability curves** (`plot_reliability`): Essential for probabilistic forecasts to check if the forecast probability matches the observed frequency.
* **Bar plots** (`plot_bar_bss_rpss_auc`): Best for comparing summary scores like Brier Skill Score (BSS) or Area Under Curve (AUC) across different model versions.

#### Best practices for output

1.  **Storage space**: Saving spatial metrics as NetCDF can consume significant disk space if you are working with high-resolution global data over many years.
2.  **Workflow efficiency**: During the initial setup or debugging phase, it is often helpful to set the `plot_...` keys to `false` to speed up the run. Once the logic is verified, switch them on for the final run.
3.  **Directory creation**: HCWF creates the folders if it has permission to.

<br>

---
### 11. Blending

A `blend:` section turns the run into the training of an **onset blend**: probabilities that the onset falls in week 1, 2, 3, 4 or later, from the climatology and each model's forecast onsets. It uses the sections above (observations, models, onset, `units:`, evaluation). The keys are `years`, `formulas`, `models` (per-model lead days and rain predictors), `fixed_cutoff`, `reference_onset`, `climatology`, `cv_methods`, `baselines`, `mme`, `random_forest` and more; `hcwf-run init --blend` writes a skeleton.

```yaml
blend:
  years: {training: '2019:2023', cv_holdout: '2019:2023', true_holdout: 2024}
  formulas: {blended_model: 'outcome ~ prob_clim_qx + diff_IFS_qx'}
  models:
    IFS: {lead_days: [1, 40], rain_predictors: [{agg: diff}]}
```

Train with `hcwf-run -p my_config.yaml` and apply the trained blend to live forecasts with `hcwf-run forecast runs/<run>/blend/trained.blend --forecast IFS=live/ifs.nc`. See `docs/blending.md` for the keys, the outputs and the forecast contract, and `docs/migrating_from_onset_blending.md` if you have an onset_blending config.